# Transition → Oscillation → Accumulation smoke test

This deterministic notebook validates the complete behavioral construction chain without external datasets.

In [ ]:
import pandas as pd

import featuregraph as fg

In [ ]:
observations = pd.DataFrame(
    {
        "sequence": [1] * 11,
        "signal": [0.0, 1.0, 2.0, 2.0, 1.0, 0.0, 1.0, 2.0, 2.0, 1.0, 0.0],
    }
)

In [ ]:
oscillation = fg.oscillation.Oscillation(
    signals="signal",
    group="sequence",
    diff_lag=1,
)
features = oscillation.fit_transform(observations)
oscillations = oscillation.summarize(features, "signal")

In [ ]:
transition = fg.transition.Transition(
    signals="signal",
    group="sequence",
    diff_lag=1,
)
transitions = transition.summarize(features, "signal")

assert set(transitions.table["direction"]) == {"rising", "falling", "inactive"}
assert transitions.table["is_complete"].all()

In [ ]:
accumulation = fg.accumulation.Accumulation(
    signals="signal",
    group="sequence",
    threshold="min",
)
accumulation_features = accumulation.fit_transform(features)
accumulations = accumulation.summarize(accumulation_features, "signal")

In [ ]:
assert oscillations.table["oscillation_id"].tolist() == [1]
assert accumulations.table["accumulation_id"].tolist() == [1]
assert oscillations.table.loc[0, "start_index"] == 0
assert oscillations.table.loc[0, "peak_index"] == 2
assert oscillations.table.loc[0, "end_index"] == 5
assert accumulations.table.loc[0, "is_complete"]